# Task 3: End-to-End ABSA with PhoBERT
## Unified Model: Extraction + Classification in ONE model

**Evaluation** (thong nhat voi NB03, NB05, phobert-crf-absa.ipynb):
- Span-Level Exact Match F1
- Sentence-Level Multi-Label F1 (Micro/Macro/Weighted + per-label)


## 1. Setup


In [1]:
import subprocess, sys, os

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'underthesea', 'pytorch-crf', 'transformers', 'py_vncorenlp'])

IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    DATA_DIR = '/kaggle/input/datasets/danghoang1302/uit-visd4sa'
    SAVE_DIR = '/kaggle/working/results/e2e'
    SRC_INPUT = '/kaggle/input/datasets/danghoang1302/absa-src'
    os.system(f'cp -r {SRC_INPUT}/src /kaggle/working/src')
    sys.path.insert(0, '/kaggle/working')
    print(f"KAGGLE | Data: {DATA_DIR}")
else:
    sys.path.insert(0, os.path.abspath(".."))
    DATA_DIR = os.path.join("..", "..", "data")
    SAVE_DIR = os.path.join("..", "..", "results", "e2e")
    print(f"LOCAL mode")

os.makedirs(SAVE_DIR, exist_ok=True)
for fn in ['train.jsonl', 'dev.jsonl', 'test.jsonl']:
    assert os.path.exists(os.path.join(DATA_DIR, fn)), f"MISSING: {fn}"
print("All data files OK!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.5 MB/s eta 0:00:00
KAGGLE | Data: /kaggle/input/datasets/danghoang1302/uit-visd4sa
All data files OK!


## 2. Imports & Load Data


In [2]:
import torch, torch.nn as nn, numpy as np, pandas as pd
from torch.utils.data import DataLoader
from IPython.display import display

from src.utils.preprocess import load_raw_data
from src.e2e.e2e_dataset import E2EDataset, BIO_TAGS, TAG2ID, NUM_TAGS, LABEL_NAMES
from src.e2e.e2e_model import E2EPhoBertCRF
from src.utils.engine import train_e2e_model, predict_e2e
from src.utils.metrics import (bio_tags_to_spans, evaluate_spans_f1, token_accuracy,
                               bio_to_sentence_labels, evaluate_multilabel)
from src.utils.visualization import plot_training_curves, plot_model_comparison

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42); np.random.seed(42)
print(f"Device: {device}")

train_items = load_raw_data(os.path.join(DATA_DIR, "train.jsonl"))
dev_items = load_raw_data(os.path.join(DATA_DIR, "dev.jsonl"))
test_items = load_raw_data(os.path.join(DATA_DIR, "test.jsonl"))
print(f"Train: {len(train_items)} | Dev: {len(dev_items)} | Test: {len(test_items)}")


--2026-04-26 13:21:43--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 4.08M 6s
    50K .......... .......... .......... .......... ..........  0% 13.7M 4s
   100K .......... .......... .......... .......... ..........  0% 6.70M 4s
   150K .......... .......... .......... .......... ..........  0% 24.7M 3s
   200K .......... .......... .......... .......... ..........  0% 28.5M 3s
   250K .......... .......... .......... .......... ..........  1% 7.71M 3s
   300K .......... .......... .......... .......... ..........  1% 52.7M 3s
   350K ..

2026-04-26 13:21:47 INFO  WordSegmenter:24 - Loading Word Segmentation model
Device: cuda
Train: 7785 | Dev: 1112 | Test: 2225


## 3. E2E Dataset & Train


In [3]:
MAX_LEN = 256
BATCH_SIZE = 16

print("Tokenizing with PhoBERT...")
train_ds = E2EDataset(train_items, max_len=MAX_LEN)
dev_ds = E2EDataset(dev_items, max_len=MAX_LEN)
test_ds = E2EDataset(test_items, max_len=MAX_LEN)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, BATCH_SIZE)
test_loader = DataLoader(test_ds, BATCH_SIZE)
print(f"Ready! Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)}")


Tokenizing with PhoBERT...


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing 7785 items...
Tokenizing 1112 items...
Tokenizing 2225 items...
Ready! Train: 7785 | Dev: 1112 | Test: 2225


In [4]:
LR_OPTIONS = [2e-5, 5e-5]
all_models, all_histories, all_results = {}, {}, []

for lr in LR_OPTIONS:
    name = f"PhoBERT-CRF_lr{lr}"
    model = E2EPhoBertCRF(num_unified_tags=NUM_TAGS, dropout=0.3).to(device)
    model, history = train_e2e_model(
        model, train_loader, dev_loader, device,
        lr=lr, epochs=15, patience=5, model_name=name)
    all_models[name] = model
    all_histories[name] = history


pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]


  Training PhoBERT-CRF_lr2e-05 (PhoBERT) - Differential LR
  BERT LR: 2e-05, Head LR: 1e-3
  Batch: 16 x 2 accum = 32 effective
  Total steps: 3652, Warmup: 365


...

  Ep 13/15 | Train Loss: 1.8674 | Dev Loss: 35.9911 | TokAcc: 0.7543 | SpanF1: 0.5771 | LR: 7.30e-06 | 509.9s


Training:   0%|          | 0/487 [00:00<?, ?it/s]

  Ep 14/15 | Train Loss: 1.6148 | Dev Loss: 36.5890 | TokAcc: 0.7547 | SpanF1: 0.5763 | LR: 3.59e-06 | 507.5s


Training:   0%|          | 0/487 [00:00<?, ?it/s]

  Ep 15/15 | Train Loss: 1.4359 | Dev Loss: 37.0271 | TokAcc: 0.7519 | SpanF1: 0.5766 | LR: 0.00e+00 | 508.9s


## 4. Evaluate on Test Set


In [5]:
best_f1_global, best_model_key, best_mt, best_test_res = 0, "", None, None

for name, model in all_models.items():
    test_res = predict_e2e(model, test_loader, device)
    pred_spans = [bio_tags_to_spans(pt, BIO_TAGS, l) for pt, l in zip(test_res['pred_tags'], test_res['lengths'])]
    true_spans = [bio_tags_to_spans(tt, BIO_TAGS, l) for tt, l in zip(test_res['true_tags'], test_res['lengths'])]
    span_f1 = evaluate_spans_f1(pred_spans, true_spans)

    pred_sent, _ = bio_to_sentence_labels(test_res['pred_tags'], test_res['lengths'], BIO_TAGS, LABEL_NAMES)
    true_sent, _ = bio_to_sentence_labels(test_res['true_tags'], test_res['lengths'], BIO_TAGS, LABEL_NAMES)
    mt = evaluate_multilabel(true_sent, pred_sent, LABEL_NAMES)

    all_results.append({"Model": name, "Span_F1": round(span_f1['f1'], 4),
        "Micro_F1": round(mt['micro']['f1'], 4), "Macro_F1": round(mt['macro']['f1'], 4)})
    print(f"{name} | Span F1: {span_f1['f1']:.4f} | Micro F1: {mt['micro']['f1']:.4f}")

    if span_f1['f1'] > best_f1_global:
        best_f1_global = span_f1['f1']
        best_model_key = name
        best_mt = mt
        best_test_res = test_res
        best_pred_spans = pred_spans
        best_true_spans = true_spans

e2e_df = pd.DataFrame(all_results)
display(e2e_df)

print(f"\n  BEST: {best_model_key}")
print(f"  {'Label':<25} {'P':>7} {'R':>7} {'F1':>7} {'Sup':>6}")
print(f"  {'-'*55}")
for ln in LABEL_NAMES:
    m = best_mt[ln]
    print(f"  {ln:<25} {m['precision']:>7.4f} {m['recall']:>7.4f} {m['f1']:>7.4f} {m['support']:>6d}")


PhoBERT-CRF_lr2e-05 | Span F1: 0.5934 | Micro F1: 0.8529
PhoBERT-CRF_lr5e-05 | Span F1: 0.6015 | Micro F1: 0.8541


,Model,Span_F1,Micro_F1,Macro_F1
0,PhoBERT-CRF_lr2e-05,0.5934,0.8529,0.6881
1,PhoBERT-CRF_lr5e-05,0.6015,0.8541,0.6935



  BEST: PhoBERT-CRF_lr5e-05
  Label                           P       R      F1    Sup
  -------------------------------------------------------
  CAMERA#POSITIVE            0.9127  0.9364  0.9244    346
  CAMERA#NEUTRAL             0.6711  0.6800  0.6755     75
  CAMERA#NEGATIVE            0.8667  0.8914  0.8789    175
  FEATURES#POSITIVE          0.8263  0.8843  0.8543    242
  FEATURES#NEUTRAL           0.4737  0.5625  0.5143     32
  FEATURES#NEGATIVE          0.8758  0.8758  0.8758    491
  PERFORMANCE#POSITIVE       0.8931  0.9305  0.9114    691
  PERFORMANCE#NEUTRAL        0.4270  0.4935  0.4578     77
  PERFORMANCE#NEGATIVE       0.8369  0.8550  0.8458    462
  DESIGN#POSITIVE            0.9066  0.9493  0.9274    276
  DESIGN#NEUTRAL             0.5833  0.4667  0.5185     15
  DESIGN#NEGATIVE            0.6869  0.7158  0.7010     95
  PRICE#POSITIVE             0.6164  0.8033  0.6975    122
  PRICE#NEUTRAL              0.4000  0.5625  0.4675     64
  PRICE#NEGATIVE            

## 5. Demo: Test Predictions


In [6]:
import random
random.seed(42)
n_samples = min(10, len(test_items))
sample_indices = random.sample(range(len(test_items)), n_samples)

print(f"{'='*70}")
print(f"  E2E PhoBERT-CRF DEMO: {n_samples} cau test")
print(f"{'='*70}")

demo_tp, demo_fp, demo_fn = 0, 0, 0
for idx in sample_indices:
    text = test_items[idx]['text']
    words = text.split()
    true_s = best_true_spans[idx]
    pred_s = best_pred_spans[idx]
    true_set, pred_set = set(true_s), set(pred_s)
    demo_tp += len(true_set & pred_set)
    demo_fp += len(pred_set - true_set)
    demo_fn += len(true_set - pred_set)

    print(f"\n{'─'*70}")
    print(f"  [{idx}] {text[:100]}{'...' if len(text)>100 else ''}")
    print(f"  TRUE ({len(true_s)}):")
    for label, s, e in true_s:
        aspect = ' '.join(words[s:e]) if e <= len(words) else '?'
        match = 'OK' if (label, s, e) in pred_set else 'MISSED'
        print(f"    {label:<30} [{s}:{e}] \"{aspect}\"  {match}")
    print(f"  PRED ({len(pred_s)}):")
    for label, s, e in pred_s:
        aspect = ' '.join(words[s:e]) if e <= len(words) else '?'
        match = 'OK' if (label, s, e) in true_set else 'WRONG'
        print(f"    {label:<30} [{s}:{e}] \"{aspect}\"  {match}")
    if not pred_s:
        print(f"    (khong co prediction)")

print(f"\n{'='*70}")
print(f"  Demo total: TP={demo_tp}, FP={demo_fp}, FN={demo_fn}")


  E2E PhoBERT-CRF DEMO: 10 cau test

──────────────────────────────────────────────────────────────────────
  [456] Sản phẩm sài tốt game mượt, hoàn hảo chưa có vấn đề gì.nhân viên phục vụ rất tốt 👍👍😄😄
  TRUE (4):
    GENERAL#POSITIVE               [0:3] "Sản phẩm sài"  OK
    PERFORMANCE#POSITIVE           [3:5] "tốt game"  OK
    GENERAL#POSITIVE               [6:11] "hoàn hảo chưa có vấn"  MISSED
    SER&ACC#POSITIVE               [11:15] "đề gì.nhân viên phục"  MISSED
  PRED (4):
    GENERAL#POSITIVE               [0:3] "Sản phẩm sài"  OK
    PERFORMANCE#POSITIVE           [3:5] "tốt game"  OK
    GENERAL#POSITIVE               [6:10] "hoàn hảo chưa có"  WRONG
    SER&ACC#POSITIVE               [10:15] "vấn đề gì.nhân viên phục"  WRONG

──────────────────────────────────────────────────────────────────────
  [102] Sau hơn một tháng sử dụng mình đánh giá tất cả mọi thứ OK nha , không biết sao nhiều người nói nhiều...
  TRUE (1):
    GENERAL#POSITIVE               [5:12] "dụng mình đ

## 6. Save


In [7]:
os.makedirs(SAVE_DIR, exist_ok=True)
e2e_df.to_csv(os.path.join(SAVE_DIR, "e2e_results.csv"), index=False)
best_model = all_models[best_model_key]
torch.save(best_model.state_dict(), os.path.join(SAVE_DIR, "best_e2e_phobert.pt"))
print(f"Best E2E: {best_model_key} (Span F1={best_f1_global:.4f}) -> Saved!")


Best E2E: PhoBERT-CRF_lr5e-05 (Span F1=0.6015) -> Saved!
